# Strategy 1. Naive generation of cypher queries

These specific stategies will be implemented: 
- Just a single user prompt
- System prompt that describes the structure of the database + user prompt with natural question

We will test this against questions from the list. The queries needs to be run separately

In [2]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [3]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic

def ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    elif re.search(r"mistral", model):
        return ChatMistralAI(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

# models:
#   claude-3-5-sonnet-20240620
#   gpt-4o
#   o1-preview-2024-09-12
#   open-mistral-7b

# llm = ChatModel("gpt-4o")


In [4]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)



In [5]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [6]:
# convenience functions for data retrieval from graph

import time

def query_llm(llm_model, prompt_template, question):
    llm = ChatModel(model = llm_model)
    prompt = prompt_template.invoke({"question": question})
    result = llm.invoke(prompt)
    return result.content

def query_cypher_graph(graph, query):
    return graph.query(query)


def query_graph(llm_output):
    cyphers = extract_cypher(llm_output)
    cypher_results = []
    for query in cyphers:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def evaluate_query_single_prompt(llm_model, prompt_template, question):

    llm_output = query_llm(llm_model, prompt_template, question)
    cypher_results = query_graph(llm_output)
    return {
        "model": llm_model,
        "question": question,
        "llm_answer": llm_output,
        "cypher_output": cypher_results
    }


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0]['query'],
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

In [7]:
questions = [
    "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)",
    "What is the evidence linking TDP-43 to cancer in animal models?",
    "What (or is there) is the clinical evidence linking BRAF to Melanoma?"
]

## Single user prompt

In [84]:
from langchain_core.prompts import PromptTemplate

template_query = """
I have a Neo4j graph with biological data that was extracted from OpenTargets using biocypher. 

Generate a cypher query that would help me answer the following scientific question:
{question}

"""

template = PromptTemplate.from_template(template_query)


Testing it with some AIs

In [85]:
models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]

# Since it's such a stupid strategy, we will not going to spend much resources on it. Just 1 iteration should be enough

todo = [(m, q) for m in models for q in questions]

# Making it slightly more complicated than it needs to be because sometime graph hangs, and 
# we don't want to lose precious strawberry output because of it

llm_answers = []
for model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(query_llm(model, template, question))


Prompting LLM: 100%|██████████| 12/12 [02:32<00:00, 12.68s/it]


In [102]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))


Querying graph: 100%|██████████| 12/12 [00:02<00:00,  5.24it/s]


In [111]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("01a-evaluations.xlsx", index=False)
with open("01a-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",To answer the scientific question about the ev...,[{'query': 'MATCH (protein:Protein {name: 'TDP...,1,MATCH (protein:Protein {name: 'TDP-43'})-[:ASS...,True,[],0.101660,0.0,NaN
1,gpt-4o,What is the evidence linking TDP-43 to cancer ...,To answer the question of what evidence links ...,[{'query': 'MATCH (gene:Gene {name: 'TDP-43'})...,1,MATCH (gene:Gene {name: 'TDP-43'})-[:ASSOCIATE...,True,[],0.096047,0.0,NaN
2,gpt-4o,What (or is there) is the clinical evidence li...,To answer the question of whether there is cli...,"[{'query': 'MATCH (g:Gene {name: ""BRAF""})-[:AS...",1,"MATCH (g:Gene {name: ""BRAF""})-[:ASSOCIATED_WIT...",True,[],0.097722,0.0,NaN
3,claude-3-5-sonnet-20240620,"What (or how strong, or is there any) is the e...",To investigate the evidence between TDP-43 and...,[{'query': 'MATCH (protein:Protein)-[:ASSOCIAT...,2,MATCH (protein:Protein)-[:ASSOCIATED_WITH]->(g...,True,[],0.096167,0.0,NaN
4,claude-3-5-sonnet-20240620,What is the evidence linking TDP-43 to cancer ...,"To answer the scientific question ""What is the...",[{'query': 'MATCH (protein:Protein {symbol: 'T...,1,MATCH (protein:Protein {symbol: 'TARDBP'})-[:A...,True,[],0.100411,0.0,NaN
5,claude-3-5-sonnet-20240620,What (or is there) is the clinical evidence li...,To explore the clinical evidence linking BRAF ...,[{'query': 'MATCH (gene:Gene {symbol: 'BRAF'})...,2,MATCH (gene:Gene {symbol: 'BRAF'})-[:ASSOCIATE...,False,NaN,NaN,NaN,[Statement.SyntaxError] Type mismatch: associa...
6,open-mistral-7b,"What (or how strong, or is there any) is the e...","To answer your question using Cypher, the quer...",[{'query': 'MATCH (protein:Protein {name: 'TDP...,1,MATCH (protein:Protein {name: 'TDP-43'})-[evid...,False,NaN,NaN,NaN,[Statement.SyntaxError] Type mismatch: expecte...
7,open-mistral-7b,What is the evidence linking TDP-43 to cancer ...,To answer your scientific question using Neo4j...,[{'query': 'MATCH (tdp:Protein {name: 'TDP-43'...,1,MATCH (tdp:Protein {name: 'TDP-43'})-[:INVOLVE...,True,[],0.100936,0.0,NaN
8,open-mistral-7b,What (or is there) is the clinical evidence li...,"To answer your question using Cypher, the quer...",[{'query': 'MATCH (gene:Gene {name: 'BRAF'})-[...,1,MATCH (gene:Gene {name: 'BRAF'})-[:IS_ASSOCIAT...,True,[],0.096054,0.0,NaN
9,o1-preview-2024-09-12,"What (or how strong, or is there any) is the e...",Certainly! To find out the evidence between TD...,[{'query': 'MATCH (t:Target {approved_symbol: ...,4,MATCH (t:Target {approved_symbol: 'TARDBP'})-[...,True,[],0.096473,0.0,NaN


## Prompt with KG description

We are providing what are different types of nodes and relationships in KG, but do not explicitly tell how to query

In [113]:
from langchain_core.prompts import PromptTemplate

template_query = """
I have a Neo4j graph with biological data that was extracted from OpenTargets using biocypher. 

My graph has nodes with label Gene, and property "approvedSymbol" which is an hgnc approved symbol, and diseases with property "name" which is approved disease name. These are linked to "GeneToDiseaseAssociation" nodes via relationships IS_PART_OF. Note that disease name is taken from ontology, so should be interpreted with some flexibility (e.g. I don't know if it's all lower case or sentence case).

Generate a cypher query that would help me answer the following scientific question:
{question}

"""

template = PromptTemplate.from_template(template_query)

In [114]:
models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for m in models for q in questions for _ in range(niter)]

llm_answers = []
for model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(query_llm(model, template, question))


Prompting LLM: 100%|██████████| 120/120 [23:40<00:00, 11.84s/it]


In [156]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph: 100%|██████████| 120/120 [01:49<00:00,  1.10it/s]


In [157]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("01b-evaluations.xlsx", index=False)
with open("01b-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",To address your scientific question regarding ...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'TD...,1,MATCH (g:Gene {approvedSymbol: 'TDP-43'})-[:IS...,True,[],0.149825,0.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...",To address your scientific question regarding ...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'TD...,2,MATCH (g:Gene {approvedSymbol: 'TDP-43'})-[:IS...,True,[],0.149279,0.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...",To answer the question about the evidence betw...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'TA...,1,MATCH (g:Gene {approvedSymbol: 'TARDBP'})-[:IS...,True,[],0.150407,0.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...",To answer your scientific question about the e...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'TD...,1,MATCH (g:Gene {approvedSymbol: 'TDP-43'})-[:IS...,True,[],0.133933,0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",To find the evidence between the gene TDP-43 a...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'TD...,1,MATCH (g:Gene {approvedSymbol: 'TDP-43'})-[:IS...,True,[],0.147678,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking the gene...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'BR...,2,MATCH (g:Gene {approvedSymbol: 'BRAF'})-[]-(a:...,True,[],0.134004,0.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"To answer the scientific question, ""What is th...",[{'query': 'MATCH (g:Gene {approvedSymbol: 'BR...,4,MATCH (g:Gene {approvedSymbol: 'BRAF'})-[:IS_P...,True,"[({'biotype': 'protein_coding', 'licence': 'ht...",2.668975,5847.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking **BRAF**...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'BR...,2,MATCH (g:Gene {approvedSymbol: 'BRAF'})-[:IS_P...,True,[],0.134068,0.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking the gene...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'BR...,2,MATCH (g:Gene {approvedSymbol: 'BRAF'})-[:IS_P...,True,[],0.133145,0.0,NaN


## Prompt with additional KG description

A common failure mode is inability to correctly specify the directionality of relationships (e.g. using gene -> association -> disease instead of geen -> association <- disease). We'll try to mitigate by explicitly telling LLM to ignore relationship directionality

In [8]:
from langchain_core.prompts import PromptTemplate

template_query = """
I have a Neo4j graph with biological data that was extracted from OpenTargets using biocypher. 

My graph has nodes with label Gene, and property "approvedSymbol" which is an hgnc approved symbol, and diseases with property "name" which is approved disease name. These are linked to "GeneToDiseaseAssociation" nodes via relationships IS_PART_OF. Note that disease name is taken from ontology, so should be interpreted with some flexibility (e.g. I don't know if it's all lower case or sentence case). Note: in your cypher queries please ignore the directionality of relationships.

Generate a cypher query that would help me answer the following scientific question:
{question}

"""

template = PromptTemplate.from_template(template_query)

In [13]:
models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]

llm_answers = []
for model, question in tqdm(todo, desc="Prompting LLM"):
    try:
        llm_answers.append(query_llm(model, template, question))
    except Exception as e:
        print(e)
        llm_answers.append(None)

Prompting LLM:  42%|████▏     | 50/120 [08:03<11:16,  9.67s/it]


NameError: name 'HTTPStatusError' is not defined

In [12]:
model

'open-mistral-7b'